# Backtest Framework Comparsion

This notebook compares the same 50/200 SMA crossover strategy across `backtesting.py`, Zipline Reloaded, and NautilusTrader. 
It runs the strategy twice per framework, once on `yfinance` data and once on `fmp` data, so the table shows both financial performance and runtime.

The last section decomposes the outcome into a simple two-factor view so you can see whether the data vendor or the backtest framework moves results more.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import subprocess
import sys
from time import perf_counter

import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
if not (repo_root / 'quant_orchestrator').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quant_orchestrator.platforms.backtesting_frameworks.shared import load_signal_frame
from quant_orchestrator.platforms.backtesting_frameworks.backtesting_py.sma_crossover import (
    run_sma_crossover_backtest as run_backtesting_py,
)
from quant_orchestrator.platforms.backtesting_frameworks.zipline.sma_crossover import (
    run_sma_crossover_backtest as run_zipline,
)

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

SYMBOL = 'AAPL'
PROVIDERS = ('yfinance', 'fmp')
FRAMEWORKS = {
    'backtesting.py': run_backtesting_py,
    'zipline': run_zipline,
}
FAST_WINDOW = 50
SLOW_WINDOW = 200
CAPITAL_BASE = 100_000.0
START = '2020-01-01'
END = None


In [2]:
def run_nautilus_isolated(provider: str) -> pd.DataFrame:
    code = f'''from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / "quant_orchestrator").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quant_orchestrator.platforms.backtesting_frameworks.shared import load_signal_frame
from quant_orchestrator.platforms.backtesting_frameworks.nautilus.sma_crossover import run_sma_crossover_backtest

prices = load_signal_frame({SYMBOL!r}, provider={provider!r}, start={START!r}, end={END!r})
_, summary, _ = run_sma_crossover_backtest(
    prices.loc[:, ["open", "high", "low", "close", "volume"]],
    symbol={SYMBOL!r},
    fast_window={FAST_WINDOW},
    slow_window={SLOW_WINDOW},
    capital_base={CAPITAL_BASE},
)
row = summary.iloc[0].to_dict()
row['framework'] = 'nautilus'
row['provider'] = {provider!r}
row['performance_score'] = float(row['total_return']) + float(row['max_drawdown'])
clean = {{}}
for key, value in row.items():
    if hasattr(value, 'item'):
        try:
            value = value.item()
        except Exception:
            pass
    if hasattr(value, 'isoformat'):
        try:
            value = value.isoformat()
        except Exception:
            pass
    clean[key] = value
print(json.dumps(clean, default=str))
'''
    completed = subprocess.run(
        [sys.executable, '-c', code],
        check=True,
        capture_output=True,
        text=True,
        cwd=repo_root,
    )
    payload = json.loads(completed.stdout.strip().splitlines()[-1])
    return pd.DataFrame([payload])


def run_one(framework: str, provider: str) -> tuple[pd.DataFrame, pd.Series]:
    if framework == 'nautilus':
        result = run_nautilus_isolated(provider)
        equity = pd.Series(dtype=float)
        return result, equity

    prices = load_signal_frame(SYMBOL, provider=provider, start=START, end=END)
    runner = FRAMEWORKS[framework]
    _, summary, equity = runner(
        prices.loc[:, ['open', 'high', 'low', 'close', 'volume']],
        symbol=SYMBOL,
        fast_window=FAST_WINDOW,
        slow_window=SLOW_WINDOW,
        capital_base=CAPITAL_BASE,
    )
    row = summary.iloc[0].to_dict()
    row['framework'] = framework
    row['provider'] = provider
    row['performance_score'] = float(row['total_return']) + float(row['max_drawdown'])
    return pd.DataFrame([row]), equity

rows = []
equities = {}
for provider in PROVIDERS:
    for framework in ('backtesting.py', 'zipline', 'nautilus'):
        started = perf_counter()
        result, equity = run_one(framework, provider)
        result['wall_clock_seconds'] = perf_counter() - started
        rows.append(result)
        if not equity.empty:
            equities[(framework, provider)] = equity

comparison = pd.concat(rows, ignore_index=True)
comparison = comparison.sort_values(['provider', 'framework']).reset_index(drop=True)
display(
    comparison.loc[:, [
        'provider', 'framework', 'symbol', 'bars', 'trades', 'final_equity',
        'total_return', 'max_drawdown', 'performance_score', 'elapsed_seconds', 'wall_clock_seconds', 'bars_per_second',
    ]].round(4),
)


def factor_decomposition(table: pd.DataFrame, metric: str) -> pd.DataFrame:
    pivot = table.pivot(index='framework', columns='provider', values=metric)
    grand_mean = float(pivot.to_numpy(dtype=float).mean())
    framework_means = pivot.mean(axis=1)
    provider_means = pivot.mean(axis=0)
    framework_ss = float(len(provider_means) * ((framework_means - grand_mean) ** 2).sum())
    provider_ss = float(len(framework_means) * ((provider_means - grand_mean) ** 2).sum())
    total_ss = float(((pivot - grand_mean) ** 2).to_numpy().sum())
    residual_ss = max(0.0, total_ss - framework_ss - provider_ss)
    total = framework_ss + provider_ss + residual_ss
    return pd.DataFrame([
        {
            'metric': metric,
            'framework_ss': framework_ss,
            'provider_ss': provider_ss,
            'residual_ss': residual_ss,
            'framework_share': framework_ss / total if total else 0.0,
            'provider_share': provider_ss / total if total else 0.0,
            'residual_share': residual_ss / total if total else 0.0,
            'dominant_factor': 'provider' if provider_ss > framework_ss else 'framework',
            'provider_to_framework_ratio': (provider_ss / framework_ss) if framework_ss else float('inf'),
        }
    ])


effect_report = pd.concat([
    factor_decomposition(comparison, 'performance_score'),
    factor_decomposition(comparison, 'total_return'),
    factor_decomposition(comparison, 'elapsed_seconds'),
], ignore_index=True)
display(effect_report.round(4))

framework_summary = comparison.groupby('framework').agg(
    mean_total_return=('total_return', 'mean'),
    mean_max_drawdown=('max_drawdown', 'mean'),
    mean_elapsed_seconds=('elapsed_seconds', 'mean'),
    mean_bars_per_second=('bars_per_second', 'mean'),
)
provider_summary = comparison.groupby('provider').agg(
    mean_total_return=('total_return', 'mean'),
    mean_max_drawdown=('max_drawdown', 'mean'),
    mean_elapsed_seconds=('elapsed_seconds', 'mean'),
    mean_bars_per_second=('bars_per_second', 'mean'),
)
display(framework_summary.round(4))
display(provider_summary.round(4))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/backtesting/_plotting.py:50: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support (e.g. PyCharm, Spyder IDE). Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


,provider,framework,symbol,bars,trades,final_equity,total_return,max_drawdown,performance_score,elapsed_seconds,wall_clock_seconds,bars_per_second
0,fmp,backtesting.py,AAPL,1428,5,107565.76,0.0757,-0.1485,-0.0728,0.0160,0.0445,89073.95
1,fmp,nautilus,AAPL,1428,9,106645.60,0.0665,-0.1477,-0.0812,0.0908,0.8699,15728.00
2,fmp,zipline,AAPL,1229,9,105274.06,0.0527,-0.2152,-0.1625,0.4442,0.4624,2767.00
3,yfinance,backtesting.py,AAPL,1427,5,108402.13,0.0840,-0.1448,-0.0608,0.0163,0.2758,87497.96
4,yfinance,nautilus,AAPL,1427,9,105224.68,0.0522,-0.1468,-0.0946,0.0919,0.8566,15525.13
5,yfinance,zipline,AAPL,1228,9,107639.03,0.0764,-0.2132,-0.1368,0.6618,1.4410,1855.65


,metric,framework_ss,provider_ss,residual_ss,framework_share,provider_share,residual_share,dominant_factor,provider_to_framework_ratio
0,performance_score,0.0074,0.0001,0.0004,0.9378,0.0124,0.0498,framework,0.0133
1,total_return,0.0005,0.0001,0.0004,0.5211,0.0599,0.4190,framework,0.1149
2,elapsed_seconds,0.3380,0.0080,0.0157,0.9345,0.0221,0.0434,framework,0.0237


,mean_total_return,mean_max_drawdown,mean_elapsed_seconds,mean_bars_per_second
framework,,,,
backtesting.py,0.0798,-0.1466,0.0161,88285.955
nautilus,0.0594,-0.1472,0.0914,15626.565
zipline,0.0646,-0.2142,0.5530,2311.325


,mean_total_return,mean_max_drawdown,mean_elapsed_seconds,mean_bars_per_second
provider,,,,
fmp,0.0650,-0.1705,0.1837,35856.3167
yfinance,0.0709,-0.1683,0.2567,34959.5800


In [3]:
def factor_decomposition(table: pd.DataFrame, metric: str) -> pd.DataFrame:
    pivot = table.pivot(index='framework', columns='provider', values=metric)
    grand_mean = float(pivot.to_numpy(dtype=float).mean())
    framework_means = pivot.mean(axis=1)
    provider_means = pivot.mean(axis=0)
    framework_ss = float(len(provider_means) * ((framework_means - grand_mean) ** 2).sum())
    provider_ss = float(len(framework_means) * ((provider_means - grand_mean) ** 2).sum())
    total_ss = float(((pivot - grand_mean) ** 2).to_numpy().sum())
    residual_ss = max(0.0, total_ss - framework_ss - provider_ss)
    total = framework_ss + provider_ss + residual_ss
    return pd.DataFrame([
        {
            'metric': metric,
            'framework_ss': framework_ss,
            'provider_ss': provider_ss,
            'residual_ss': residual_ss,
            'framework_share': framework_ss / total if total else 0.0,
            'provider_share': provider_ss / total if total else 0.0,
            'residual_share': residual_ss / total if total else 0.0,
            'dominant_factor': 'provider' if provider_ss > framework_ss else 'framework',
            'provider_to_framework_ratio': (provider_ss / framework_ss) if framework_ss else float('inf'),
        }
    ])

effect_report = pd.concat([
    factor_decomposition(comparison, 'performance_score'),
    factor_decomposition(comparison, 'total_return'),
    factor_decomposition(comparison, 'elapsed_seconds'),
], ignore_index=True)
display(effect_report.round(4))

framework_summary = comparison.groupby('framework').agg(
    mean_total_return=('total_return', 'mean'),
    mean_max_drawdown=('max_drawdown', 'mean'),
    mean_elapsed_seconds=('elapsed_seconds', 'mean'),
    mean_bars_per_second=('bars_per_second', 'mean'),
)
provider_summary = comparison.groupby('provider').agg(
    mean_total_return=('total_return', 'mean'),
    mean_max_drawdown=('max_drawdown', 'mean'),
    mean_elapsed_seconds=('elapsed_seconds', 'mean'),
    mean_bars_per_second=('bars_per_second', 'mean'),
)
display(framework_summary.round(4))
display(provider_summary.round(4))


,metric,framework_ss,provider_ss,residual_ss,framework_share,provider_share,residual_share,dominant_factor,provider_to_framework_ratio
0,performance_score,0.0074,0.0001,0.0004,0.9378,0.0124,0.0498,framework,0.0133
1,total_return,0.0005,0.0001,0.0004,0.5211,0.0599,0.4190,framework,0.1149
2,elapsed_seconds,0.3380,0.0080,0.0157,0.9345,0.0221,0.0434,framework,0.0237


,mean_total_return,mean_max_drawdown,mean_elapsed_seconds,mean_bars_per_second
framework,,,,
backtesting.py,0.0798,-0.1466,0.0161,88285.955
nautilus,0.0594,-0.1472,0.0914,15626.565
zipline,0.0646,-0.2142,0.5530,2311.325


,mean_total_return,mean_max_drawdown,mean_elapsed_seconds,mean_bars_per_second
provider,,,,
fmp,0.0650,-0.1705,0.1837,35856.3167
yfinance,0.0709,-0.1683,0.2567,34959.5800


The first table is the direct comparison. The second table is the comparison-of-comparisons: if `provider_ss` is larger than `framework_ss`, the vendor choice moved the strategy more than the engine choice for this setup.